# Prithvi-EO-2.0 + Sen1Floods11 BWER Paper-Prep Workflow

This notebook combines the existing Prithvi + Sen1Floods11 Colab pipeline with the BWER slice-support preflight and formal BWER audit. It is designed to decide whether event-level or country-level flood-mapping slice risk is suitable for paper-grade analysis.

The chip-level classification path is a sanity audit for consistency with earlier RSFM experiments. The segmentation path is the more deployment-relevant Sen1Floods11 audit. The first runs should be interpreted as paper-prep or pilot evidence unless support diagnostics show adequate slice coverage.


In [ ]:
# 1. Clone or update this repo
from pathlib import Path
REPO_URL = "https://github.com/strivekboy-coder/rsfm-fairness-audit.git"  # edit if you use a fork
PROJECT_ROOT = Path("/content/rsfm-fairness-audit")
%cd /content
if PROJECT_ROOT.exists():
    %cd {PROJECT_ROOT}
    !git pull
else:
    !git clone {REPO_URL} {PROJECT_ROOT}
    %cd {PROJECT_ROOT}
print('repo root:', Path.cwd())


In [ ]:
# 2. Install package and Prithvi dependencies
# Keep numpy below 2.1 because Colab's numba stack requires numpy<2.1.
!python -m pip install "numpy>=1.24,<2.1"
!python -m pip install -e .
!python -m pip install -r requirements-prithvi.txt
!python -m pip install --upgrade terratorch "numpy>=1.24,<2.1"
!python - <<'PY'
import numpy
print('numpy', numpy.__version__)
try:
    import numba
    print('numba', numba.__version__)
except Exception as exc:
    print('numba check skipped:', exc)
PY


In [ ]:
# 3. Check GPU
import torch
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


In [ ]:
# 4. Configure run paths and low-RAM controls
# Standard Colab first-pass settings. Use High-RAM by raising CHUNK_SIZE/BATCH_SIZE or enabling segmentation.
MAX_SAMPLES = 64
RUN_CLASSIFICATION = True
RUN_SEGMENTATION = False
LOW_RAM_MODE = True
CHUNK_SIZE = 8 if LOW_RAM_MODE else 32
BATCH_SIZE = 1 if LOW_RAM_MODE else 2
BOOTSTRAP_N = 50 if LOW_RAM_MODE else 100

DATA_ROOT = f"data/sen1floods11_prithvi_subset{MAX_SAMPLES}"
CLASS_OUTPUT = f"outputs/prithvi_sen1floods11_class{MAX_SAMPLES}"
SEG_OUTPUT = f"outputs/prithvi_sen1floods11_seg{MAX_SAMPLES}"
AUDIT_ROOT = f"outputs/audit/prithvi_sen1floods11_bwer{MAX_SAMPLES}"
CLASS_PREFLIGHT = f"{AUDIT_ROOT}/classification_preflight"
SEG_PREFLIGHT = f"{AUDIT_ROOT}/segmentation_preflight"
CLASS_AUDIT_TABLE = f"{AUDIT_ROOT}/classification_audit_table.csv"
SEG_AUDIT_TABLE = f"{AUDIT_ROOT}/segmentation_audit_table.csv"
FORMAL_ROOT = f"{AUDIT_ROOT}/formal_bwer"
print(DATA_ROOT, CLASS_OUTPUT, SEG_OUTPUT, AUDIT_ROOT, sep='
')
print('RUN_CLASSIFICATION', RUN_CLASSIFICATION, 'RUN_SEGMENTATION', RUN_SEGMENTATION)
print('LOW_RAM_MODE', LOW_RAM_MODE, 'CHUNK_SIZE', CHUNK_SIZE, 'BATCH_SIZE', BATCH_SIZE, 'BOOTSTRAP_N', BOOTSTRAP_N)


In [ ]:
# 4b. Apply notebook batch-size control to the Prithvi config
from pathlib import Path
import yaml
config_path = Path("configs/models/prithvi.yaml")
config = yaml.safe_load(config_path.read_text())
config["batch_size"] = int(BATCH_SIZE)
config["device"] = "auto"
config_path.write_text(yaml.safe_dump(config, sort_keys=False))
print(config_path.read_text())


In [ ]:
# 5. Prepare Sen1Floods11 samples directly in Colab local storage
!python scripts/prepare_sen1floods11_subset.py   --output-dir {DATA_ROOT}   --max-samples {MAX_SAMPLES}   --candidate-limit 1000
!head -5 {DATA_ROOT}/metadata.csv


In [ ]:
# 6. Existing real-run preflight
!python -m rsfm_fairness_audit.cli check-real   --dataset sen1floods11   --model prithvi   --model-config configs/models/prithvi.yaml   --data-root {DATA_ROOT}


In [ ]:
# 7. Chip-level classification sanity audit
import gc, subprocess
try:
    import torch
except Exception:
    torch = None

if RUN_CLASSIFICATION:
    cmd = [
        "python", "-m", "rsfm_fairness_audit.cli", "run-real",
        "--dataset", "sen1floods11",
        "--model", "prithvi",
        "--dataset-root", DATA_ROOT,
        "--config", "configs/models/prithvi.yaml",
        "--output-dir", CLASS_OUTPUT,
        "--max-samples", str(MAX_SAMPLES),
        "--chunk-size", str(CHUNK_SIZE),
        "--streaming-embeddings", "true",
    ]
    print(" ".join(cmd))
    subprocess.check_call(cmd)
else:
    print("Skipping classification because RUN_CLASSIFICATION=False")

gc.collect()
if torch is not None and torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# 8. Lightweight segmentation fairness audit
import gc, subprocess
try:
    import torch
except Exception:
    torch = None

if RUN_SEGMENTATION:
    cmd = [
        "python", "-m", "rsfm_fairness_audit.cli", "run-segmentation-real",
        "--dataset", "sen1floods11",
        "--model", "prithvi",
        "--dataset-root", DATA_ROOT,
        "--config", "configs/models/prithvi.yaml",
        "--output-dir", SEG_OUTPUT,
        "--max-samples", str(MAX_SAMPLES),
    ]
    print(" ".join(cmd))
    subprocess.check_call(cmd)
else:
    print("Skipping segmentation because RUN_SEGMENTATION=False. Enable it after classification+BWER works on your runtime.")

gc.collect()
if torch is not None and torch.cuda.is_available():
    torch.cuda.empty_cache()


## What `preflight-bwer` Checks

`preflight-bwer` checks whether a candidate BWER configuration has enough slice support before formal BWER is run. For example, `BWER(event_id | class_label)` asks whether every event slice has enough samples and whether the event x class_label support matrix is dense enough for balanced tail-risk analysis.

Sparse slice x balance support can make balanced BWER unstable. With `renormalize`, each slice uses only the balance levels observed inside that slice, so slices may no longer be composition-comparable. With `overlap`, only balance levels present in all valid slices are used, which can remove much of the task. The support preflight helps decide whether balanced BWER is recommended, caution-only, or not recommended.


In [ ]:
# 9. Build normalized audit tables and inspect available columns
from pathlib import Path
import pandas as pd
from rsfm_fairness_audit.audit_table import (
    build_audit_table_from_predictions,
    build_audit_table_from_segmentation_metrics,
    write_audit_table,
)

Path(AUDIT_ROOT).mkdir(parents=True, exist_ok=True)
metadata_path = Path(DATA_ROOT) / "metadata.csv"
class_rows = []
seg_rows = []
if RUN_CLASSIFICATION and (Path(CLASS_OUTPUT) / "predictions.csv").exists():
    class_rows = build_audit_table_from_predictions(
        Path(CLASS_OUTPUT) / "predictions.csv",
        metadata_path=metadata_path,
        dataset="sen1floods11",
        model="prithvi",
        task="classification",
    )
if RUN_SEGMENTATION and (Path(SEG_OUTPUT) / "segmentation_metrics.csv").exists():
    seg_rows = build_audit_table_from_segmentation_metrics(
        Path(SEG_OUTPUT) / "segmentation_metrics.csv",
        metadata_path=metadata_path,
        dataset="sen1floods11",
        model="prithvi",
        task="segmentation",
    )

def enrich_sen1_rows(rows, segmentation=False):
    enriched = []
    for row in rows:
        item = dict(row)
        if not item.get("event_id") and item.get("event"):
            item["event_id"] = item["event"]
        if segmentation:
            item.setdefault("class_label", "water")
            item.setdefault("flood_label", item.get("class_label", "water"))
        else:
            item.setdefault("flood_label", str(item.get("label", item.get("class_label", ""))))
        enriched.append(item)
    return enriched

class_rows = enrich_sen1_rows(class_rows, segmentation=False)
seg_rows = enrich_sen1_rows(seg_rows, segmentation=True)
if class_rows:
    write_audit_table(CLASS_AUDIT_TABLE, class_rows)
if seg_rows:
    write_audit_table(SEG_AUDIT_TABLE, seg_rows)

for name, path in [("classification", CLASS_AUDIT_TABLE), ("segmentation", SEG_AUDIT_TABLE)]:
    path = Path(path)
    if not path.exists():
        print(f"Skipping {name}: {path} not found")
        continue
    df = pd.read_csv(path)
    print(f"
=== {name} audit table: {path} ===")
    print("rows", len(df))
    print("columns", list(df.columns))
    display(df.head())


In [ ]:
# 10. Choose candidate BWER configurations from available metadata
PRIORITY_CANDIDATES = [
    ("event_id", "class_label"),
    ("country", "class_label"),
    ("month", "class_label"),
    ("season", "class_label"),
    ("biome", "class_label"),
    ("ecoregion", "class_label"),
    ("event_id", "flood_label"),
    ("country", "flood_label"),
    ("event_id", None),
    ("country", None),
    ("season", None),
]

def available_candidates(csv_path):
    path = Path(csv_path)
    if not path.exists():
        return [], [("", "", "audit table missing")]
    df = pd.read_csv(path, nrows=1)
    columns = set(df.columns)
    out = []
    skipped = []
    for slice_var, balance_var in PRIORITY_CANDIDATES:
        if slice_var not in columns:
            skipped.append((slice_var, balance_var, "missing slice column"))
            continue
        if balance_var and balance_var not in columns:
            skipped.append((slice_var, balance_var, "missing balance column"))
            continue
        out.append(f"{slice_var}|{balance_var}" if balance_var else slice_var)
    return out, skipped

CLASS_CANDIDATES, CLASS_SKIPPED = available_candidates(CLASS_AUDIT_TABLE)
SEG_CANDIDATES, SEG_SKIPPED = available_candidates(SEG_AUDIT_TABLE)
print("classification candidates", CLASS_CANDIDATES)
print("classification skipped", CLASS_SKIPPED)
print("segmentation candidates", SEG_CANDIDATES)
print("segmentation skipped", SEG_SKIPPED)


In [ ]:
# 11. Run BWER support preflight for classification and segmentation
import subprocess

def run_preflight(audit_table, output_dir, candidates, task_name):
    if not candidates or not Path(audit_table).exists():
        print(f"Skipping {task_name} preflight: no audit table or candidates")
        return
    cmd = [
        "python", "-m", "rsfm_fairness_audit.cli", "preflight-bwer",
        "--audit-table", str(audit_table),
        "--dataset", "sen1floods11",
        "--model", "prithvi",
        "--task", task_name,
        "--output-dir", str(output_dir),
        "--min-samples-per-slice", "1",
        "--min-units-required", "1",
    ]
    for candidate in candidates:
        cmd.extend(["--candidate", candidate])
    print(" ".join(cmd))
    subprocess.check_call(cmd)

run_preflight(CLASS_AUDIT_TABLE, CLASS_PREFLIGHT, CLASS_CANDIDATES, "classification")
run_preflight(SEG_AUDIT_TABLE, SEG_PREFLIGHT, SEG_CANDIDATES, "segmentation")

for name, out_dir in [("classification", CLASS_PREFLIGHT), ("segmentation", SEG_PREFLIGHT)]:
    rec_path = Path(out_dir) / "slice_support_recommendations.csv"
    if rec_path.exists():
        print(f"
=== {name} support recommendations ===")
        recs = pd.read_csv(rec_path)
        display(recs[["candidate", "recommendation", "preferred_bwer", "n_slices_valid", "missing_slice_balance_ratio", "reason"]])


## Reading the Preflight Recommendations

`recommended` means the candidate has enough slice support for the requested raw or balanced BWER design. `caution` means the candidate can be useful as pilot evidence or sensitivity analysis, but sparse slice x balance support may make the balanced estimate unstable. `not_recommended` means the candidate should not be used as a main paper-grade BWER result without more data or a different slice definition.

For Sen1Floods11, event-level tail risk asks whether the worst flood events have excess segmentation risk relative to typical events. Country-level tail risk is only meaningful when country metadata is verified and has enough support. If country/month/season/biome/ecoregion columns are absent, the notebook skips them rather than inventing geography or temporal metadata.


In [ ]:
# 12. Run formal BWER only for recommended candidates; fall back to best caution candidates as pilot-only
import math

def parse_candidate(text):
    inner = text.replace("BWER(", "").rstrip(")")
    if " | " in inner:
        left, right = inner.split(" | ", 1)
        return left, right
    return inner, None

def select_candidates(preflight_dir, max_caution=2):
    rec_path = Path(preflight_dir) / "slice_support_recommendations.csv"
    if not rec_path.exists():
        return [], "pilot"
    recs = pd.read_csv(rec_path)
    recommended = recs[recs["recommendation"] == "recommended"].copy()
    if len(recommended):
        return recommended["candidate"].tolist(), "paper"
    caution = recs[recs["recommendation"] == "caution"].copy()
    if len(caution):
        caution["_missing"] = pd.to_numeric(caution["missing_slice_balance_ratio"], errors="coerce").fillna(0.0)
        caution["_valid"] = pd.to_numeric(caution["n_slices_valid"], errors="coerce").fillna(0)
        caution = caution.sort_values(["_valid", "_missing"], ascending=[False, True]).head(max_caution)
        return caution["candidate"].tolist(), "pilot"
    return [], "pilot"

def run_formal_bwer(audit_table, preflight_dir, task_name):
    selected, level = select_candidates(preflight_dir)
    print(f"{task_name}: selected {selected} as audit_level={level}")
    outputs = []
    for candidate in selected:
        slice_var, balance_var = parse_candidate(candidate)
        safe_name = candidate.replace("BWER(", "").replace(")", "").replace(" | ", "__").replace(" ", "_")
        out_dir = Path(FORMAL_ROOT) / task_name / safe_name
        cmd = [
            "python", "-m", "rsfm_fairness_audit.cli", "evaluate-bwer",
            "--audit-table", str(audit_table),
            "--dataset", "sen1floods11",
            "--model", "prithvi",
            "--task", task_name,
            "--slice-variable", slice_var,
            "--output-dir", str(out_dir),
            "--missing-balance-policy", "renormalize",
            "--bootstrap", str(BOOTSTRAP_N),
            "--audit-level", level,
        ]
        if balance_var:
            cmd.extend(["--balance-variable", balance_var])
        print(" ".join(cmd))
        subprocess.check_call(cmd)
        outputs.append(out_dir)
    if not outputs:
        print(f"No recommended or caution candidates for {task_name}; formal BWER skipped.")
    return outputs

CLASS_BWER_DIRS = run_formal_bwer(CLASS_AUDIT_TABLE, CLASS_PREFLIGHT, "classification")
SEG_BWER_DIRS = run_formal_bwer(SEG_AUDIT_TABLE, SEG_PREFLIGHT, "segmentation")


In [ ]:
# 13. Inspect BWER outputs and figures
from IPython.display import Image, display

for out_dir in CLASS_BWER_DIRS + SEG_BWER_DIRS:
    out_dir = Path(out_dir)
    print(f"
=== {out_dir} ===")
    for csv_name in ["bwer_summary.csv", "bwer_by_slice.csv", "support_diagnostics.csv", "bootstrap_ci.csv"]:
        path = out_dir / csv_name
        if path.exists() and path.stat().st_size > 0:
            print(csv_name)
            display(pd.read_csv(path).head(20))
    for fig_name in ["average_vs_bwer.png", "raw_vs_balanced_bwer.png", "worst_tail_slices.png", "slice_risk_heatmap.png"]:
        fig_path = out_dir / "figures" / fig_name
        if fig_path.exists():
            print(fig_name)
            display(Image(filename=str(fig_path)))


## Suggested Scientific Finding Note

After inspecting the preflight and formal BWER outputs, add only a concise interpretation to `docs/experiments/scientific_findings.md`. Do not paste raw logs or full tables there. A suitable note should say whether Sen1Floods11 event-level or country-level BWER appears supportable, which candidates were only caution/pilot, and that 64-sample smoke runs are not paper-grade conclusions.


In [ ]:
# 14. Optionally append a concise finding note to docs/experiments/scientific_findings.md
# Review the generated text before running this cell. It records interpretation only, not raw outputs.
from datetime import date
finding_path = Path("docs/experiments/scientific_findings.md")
finding_path.parent.mkdir(parents=True, exist_ok=True)
summary_lines = [
    "",
    f"## Prithvi Sen1Floods11 BWER Paper-Prep ({date.today().isoformat()})",
    "",
    f"A {MAX_SAMPLES}-sample Prithvi-EO-2.0 + Sen1Floods11 run was used to test BWER support preflight and formal audit wiring for chip-level classification and lightweight segmentation. Interpret the result as paper-prep evidence unless the support recommendations show adequate event/country support. Raw outputs remain in the Colab output directories and are not stored in this findings log.",
]
print("
".join(summary_lines))
# Uncomment to append after review:
# with finding_path.open("a", encoding="utf-8") as handle:
#     handle.write("
".join(summary_lines) + "
")
# print(f"Appended to {finding_path}")


In [ ]:
# 15. Package final BWER evidence artifacts for download
from zipfile import ZIP_DEFLATED, ZipFile
from google.colab import files

PROJECT_ROOT = Path('/content/rsfm-fairness-audit')
ZIP_PATH = PROJECT_ROOT / f"prithvi_sen1floods11_bwer{MAX_SAMPLES}_evidence.zip"
roots = [Path(AUDIT_ROOT), Path(CLASS_OUTPUT), Path(SEG_OUTPUT)]
include_names = {
    "audit_table.csv",
    "slice_support_recommendations.csv",
    "slice_support_summary.csv",
    "slice_support_report.md",
    "warnings.json",
    "bwer_summary.csv",
    "bwer_by_slice.csv",
    "support_diagnostics.csv",
    "bootstrap_ci.csv",
    "report.md",
    "segmentation_metrics.csv",
    "fairness_summary.csv",
    "raw_vs_balanced_gap.csv",
}
files_to_zip = []
for root in roots:
    if not root.exists():
        continue
    for path in root.rglob("*"):
        if path.is_file() and (path.name in include_names or "figures" in path.parts or "tables" in path.parts):
            files_to_zip.append(path)
with ZipFile(ZIP_PATH, "w", compression=ZIP_DEFLATED) as archive:
    for path in sorted(set(files_to_zip)):
        archive.write(path, path.relative_to(PROJECT_ROOT).as_posix())
print(f"Packaged {len(files_to_zip)} artifacts into {ZIP_PATH}")
files.download(str(ZIP_PATH))
